# Tutorial 19: dCDH for Marketing Pulse Campaigns

A practitioner walkthrough for measuring lift from promotional campaigns that turn on AND off across markets at staggered times. The tutorial uses the `ChaisemartinDHaultfoeuille` estimator (alias `DCDH`) - diff-diff's only estimator built for reversible (non-absorbing) treatment, where every other modern staggered estimator in the library assumes treatment is absorbing.

## 1. The Marketing Pulse Problem

Your team runs paid-promo pulses across 60 markets. Some markets ran the promo at the start of the quarter and turned it off as the campaign budget rolled to the next geo (leavers); others started untreated and switched the promo on at some point during the quarter (joiners). Leadership wants the average lift on weekly checkout sessions while the promo was on.

**Why dCDH.** This panel has *reversible* (non-absorbing) treatment in the dCDH sense: across the panel, the promo turns on in some markets and off in others - both directions appear in the same dataset. Every other modern staggered-DiD estimator in diff-diff (Callaway-Sant'Anna, Sun-Abraham, Wooldridge ETWFE, ImputationDiD, TwoStageDiD, EfficientDiD) assumes treatment is absorbing: once treated, always treated. They simply don't apply to a panel that contains leavers. dCDH does, following [de Chaisemartin & D'Haultfoeuille (2020)](https://www.aeaweb.org/articles?id=10.1257/aer.20181169) and the [dynamic companion paper](https://www.nber.org/papers/w29873).

**Scope of this tutorial.** Each market in our panel switches *at most once* during the quarter (the dCDH paper's Assumption 5, which the default analytical SE path requires). So a market is either a stable-untreated unit, a joiner that turns the promo on exactly once, a leaver that turns it off exactly once, or a stable-treated unit. dCDH does support multi-switch within-market paths (e.g., on-off-on cycles) via `drop_larger_lower=False` plus `by_path=k` for per-path effects, but that's a separate scope - see the extensions section at the end. Implementation details and any documented deviations from R's `did_multiplegt_dyn` reference live in [`docs/methodology/REGISTRY.md`](../methodology/REGISTRY.html).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from diff_diff import DCDH, generate_reversible_did_data

plt.style.use("seaborn-v0_8-whitegrid")

## 2. The Data

We'll simulate a panel that mirrors a marketing pulse campaign:

- **60 markets**, each observed for **8 weeks**
- Some markets started the quarter with the promo on and switched it off (leavers); others started untreated and switched the promo on (joiners). Each market switches exactly once during the panel - the [A5 single-switch contract](../methodology/REGISTRY.html) the analytical SE is derived under.
- Outcome: weekly checkout sessions per market, baseline ~110
- True treatment effect: **+12 sessions per market-week** when the promo is on, with mild cell-level heterogeneity around that average.

In [ ]:
raw = generate_reversible_did_data(
    n_groups=60,
    n_periods=8,
    pattern="single_switch",
    initial_treat_frac=0.4,
    treatment_effect=12.0,
    heterogeneous_effects=True,
    effect_sd=1.5,
    group_fe_sd=8.0,
    time_trend=0.5,
    noise_sd=2.0,
    seed=46,  # locked via _scratch/dcdh_tutorial/ seed-search
)
df = raw.rename(
    columns={
        "group": "market_id",
        "period": "week",
        "treatment": "promo_on",
        "outcome": "sessions",
    }
)
df["sessions"] = df["sessions"] + 100.0  # shift to a realistic baseline

print(f"Panel shape: {df.shape}")
print(f"Markets: {df['market_id'].nunique()}")
print(f"Weeks: {sorted(df['week'].unique())}")
print(f"Sessions range: [{df['sessions'].min():.0f}, {df['sessions'].max():.0f}]")

In [ ]:
# Switcher-type counts. With pattern="single_switch" every market
# switches exactly once, so we have only joiners (0 → 1) and
# leavers (1 → 0); no never-treated or always-treated markets by
# construction.
df.groupby("switcher_type").size()

In [ ]:
# Mean sessions over time, split by which direction the market
# switched. Joiners (blue) ramp up after they turn the promo on;
# leavers (red) drop off after they turn it off.
first_treat = df.groupby("market_id")["promo_on"].first()
category = df["market_id"].map(
    lambda m: "starts off, switches on (joiner)" if first_treat[m] == 0 else "starts on, switches off (leaver)"
)
df_plot = df.assign(category=category)

fig, ax = plt.subplots(figsize=(9, 5))
for label, color in [
    ("starts off, switches on (joiner)", "#1f77b4"),
    ("starts on, switches off (leaver)", "#d62728"),
]:
    weekly = df_plot[df_plot["category"] == label].groupby("week")["sessions"].mean()
    ax.plot(weekly.index, weekly.values, label=label, color=color, marker="o", linewidth=2)
ax.set_xlabel("Week")
ax.set_ylabel("Mean weekly sessions")
ax.set_title("Marketing pulses on/off across markets - outcomes by switcher type")
ax.legend(loc="upper left")
plt.show()

## 3. Fitting dCDH

`DID_M` is the headline dCDH estimator: the average across periods of two pieces:

- **DID_+** (joiners): markets switching `0 → 1` between consecutive periods, compared to *contemporaneously untreated* control cells.
- **DID_-** (leavers): markets switching `1 → 0`, compared to *contemporaneously treated* control cells.

Both pieces use only cells whose treatment status was stable across the two periods being compared - so no treated unit is ever used as a control for another treated unit. The library reports DID_+, DID_-, and their average DID_M separately, so you can see if the two halves agree.

**Where do the controls come from?** dCDH's controls are *contemporaneously stable cells*, not a permanently-untreated comparison group. A market that's untreated at week 3 and week 4 contributes a stable-untreated cell at week 4 - even if that same market eventually turns the promo on at week 5 and keeps it on through week 8. Symmetrically, a market that's been running the promo since week 1 and is still running it at week 4 contributes a stable-treated cell at week 4. This is what lets dCDH work on panels with **no permanent never-treated markets at all** - our panel has zero never-treated and zero always-treated units, only 60 switchers. The technical condition - de Chaisemartin & D'Haultfoeuille's Assumption 11 - is **per-period**: at every period when a switcher exists, at least one stable cell of the relevant type also exists. The library checks A11 at fit time period-by-period and emits a `UserWarning` (zeroing the offending period's contribution by paper convention) if any switching period lacks stable controls. A11 is *not* automatic on single-switch panels - the test suite has a single-switch panel where joiners exist at a period with zero stable-untreated controls (`tests/test_chaisemartin_dhaultfoeuille.py::TestA11Handling::test_a11_violation_zero_in_numerator_retain_in_denominator`). On the seed and DGP we use here, the fit happens not to trigger an A11 warning, so we're in the clean regime. On your own data, check the warning output before trusting the headline.

In [ ]:
model = DCDH(twfe_diagnostic=False, placebo=False, seed=42)
results = model.fit(
    df,
    outcome="sessions",
    group="market_id",
    time="week",
    treatment="promo_on",
)
print(results.summary())

**Reading the headline.** dCDH estimates the lift at **about 12.1 sessions per market-week** while the promo was on (95% CI: 11.3 to 12.8), recovering the true effect of 12.0 within sampling uncertainty. The CI half-width is about 0.7 sessions, which translates to a ~6% margin of error around a roughly 11% lift on a baseline of ~110 weekly sessions.

(We passed `placebo=False` on this fit because Phase 1's single-lag placebo SE is `NaN` by design - the per-period aggregation path doesn't have an analytical influence-function derivation. We get valid placebo CIs from the multi-horizon path in Section 4 below, which has a proper IF.)

In [ ]:
# Joiners vs leavers breakdown
jl = results.to_dataframe(level="joiners_leavers")
jl.round(3)

**Reading joiners vs leavers.** Both halves should produce a positive lift in a healthy marketing-pulse design - turning the promo on increases sessions, and turning it off decreases them. Here DID_+ ≈ 12.1 (38 joiner cells) and DID_- ≈ 11.9 (22 leaver cells): both substantially positive, both within sampling uncertainty of each other and of the true effect of 12. If they had disagreed by sign or by a large margin (say one was 5 and the other was 20), that would be a heterogeneity signal worth investigating before reporting one number to leadership.

## 4. Multi-Horizon Event Study with Bootstrap

DID_M collapses the dynamic effect to one number - the average lift across all switching cells. Setting `L_max=L` instead computes `DID_l` for each horizon `l = 1..L` after each switch, plus `DID^pl_l` placebos at horizons `l = -L..-1`. This tells you whether the on-impact lift is sustained or fades, and whether the pre-treatment placebos sit on zero.

With `L_max=2` we get two post-switch horizons and two placebo horizons. The multiplier bootstrap (`n_bootstrap=199`, matching the library's `ci_params.bootstrap` convention) gives valid CIs at every horizon, including the placebo horizons.

**About the warning you're about to see.** The fit below will emit a single `UserWarning` saying *Assumption 7 (D_{g,t} >= D_{g,1}) is violated: leavers present*. This is **expected for any reversible panel** and we don't suppress it - it's the library being explicit about a methodology choice on a separate estimand:

- **Assumption 7** is a monotonic-treatment-progression assumption used by the optional **cost-benefit delta** computation (a secondary aggregate the library reports for absorbing-treatment panels). On reversible panels the assumption fails by construction - leavers' treatment goes *down*, not up.
- The library's response is to compute the cost-benefit delta on the full sample anyway and warn that the interpretation isn't clean. The headline `DID_M`, the joiners/leavers split, and the event-study horizons are **unaffected** by this warning - they use a different aggregation that doesn't rest on A7.

So the warning is informational, points at a result we won't use in this tutorial, and is the price of admission for a reversible design. We surface it; we don't silence it.

In [ ]:
# Narrow filter: silence the spurious numpy RuntimeWarnings about
# "<...> encountered in matmul" that fire only on macOS NumPy
# builds linked against Apple's Accelerate BLAS framework.
# Accelerate sets FP error flags during matmul on certain shapes/
# values; the computation is correct (Linux / OpenBLAS users don't
# see these warnings at all). See numpy issue #26669. The filter
# is scoped to the matmul message pattern only - any unrelated
# RuntimeWarning from the fit will still surface, and the
# Assumption 7 UserWarning below is NOT suppressed (that's the
# methodology warning we explained above).
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r".*encountered in matmul",
        category=RuntimeWarning,
    )
    model_es = DCDH(
        twfe_diagnostic=False, placebo=True, n_bootstrap=199, seed=42
    )
    results_es = model_es.fit(
        df,
        outcome="sessions",
        group="market_id",
        time="week",
        treatment="promo_on",
        L_max=2,
    )

es_df = results_es.to_dataframe(level="event_study")
es_df.round(3)

In [ ]:
# Event-study errorbar plot with bootstrap CIs.
es_plot = es_df[es_df["horizon"] != 0]  # drop reference row
is_pre = es_plot["horizon"] < 0

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(
    es_plot.loc[is_pre, "horizon"],
    es_plot.loc[is_pre, "effect"],
    yerr=[
        es_plot.loc[is_pre, "effect"] - es_plot.loc[is_pre, "conf_int_lower"],
        es_plot.loc[is_pre, "conf_int_upper"] - es_plot.loc[is_pre, "effect"],
    ],
    fmt="o", color="#888888", capsize=4, label="Pre-treatment placebos",
)
ax.errorbar(
    es_plot.loc[~is_pre, "horizon"],
    es_plot.loc[~is_pre, "effect"],
    yerr=[
        es_plot.loc[~is_pre, "effect"] - es_plot.loc[~is_pre, "conf_int_lower"],
        es_plot.loc[~is_pre, "conf_int_upper"] - es_plot.loc[~is_pre, "effect"],
    ],
    fmt="o", color="#1f77b4", capsize=4, label="Post-treatment effects",
)
ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
ax.axhline(12.0, color="green", linewidth=0.8, linestyle=":", label="true effect = 12.0")
ax.set_xlabel("Weeks since promo switched")
ax.set_ylabel("Effect on weekly sessions")
ax.set_title("dCDH event study (L_max=2, multiplier bootstrap)")
ax.legend(loc="lower right")
plt.show()

**Reading the event study.**

- **Both placebo horizons** (l = -2 and l = -1) sit on zero with confidence intervals comfortably covering it. Pre-trends look parallel - we have no evidence that something other than the promo was driving session growth in the cells we're using as controls.
- **On-impact effect** at l = 1 is about **+12.4 sessions** with a 95% bootstrap CI of roughly [11.4, 13.3], covering the true effect of 12.
- **Sustained effect** at l = 2 is **+12.6 sessions** with CI [11.5, 13.6]. The lift didn't fade in the second week post-switch.

Bootstrap CIs reflect the cohort-recentered influence-function variance with the finite-sample stability the multiplier bootstrap provides. Both horizons agree closely with each other AND with the headline `DID_M` from Section 3 - a built-in consistency check across the per-period and per-group aggregation paths.

## 5. Communicating Results to Leadership

A stakeholder-ready summary of the analysis above:

> **Headline.** The pulse campaign lifted weekly checkout sessions by approximately **12 sessions per market per week** while the promo was on (95% CI: 11.3 to 12.8). On a baseline of about 110 weekly sessions per market, that's roughly an **11% lift**. *[Source: `results.overall_att` from Section 3.]*
>
> **Sample size and design.** 60 markets observed for 8 weeks (480 market-weeks). Of those, 38 markets started untreated and switched the promo on at some point during the quarter (joiners), and 22 markets started with the promo on and switched it off (leavers). Method: dCDH (de Chaisemartin & D'Haultfoeuille 2020) - diff-diff's only estimator built for treatment that can switch on AND off in the same panel. *[Source: switcher counts and panel shape from Section 2.]*
>
> **Validity evidence.** Two checks supported the result. (a) The joiners-vs-leavers split agreed: joiners produced a +12.1 lift, leavers a +11.9 lift, well within sampling uncertainty of each other and of the headline. (b) The multi-horizon placebos at l = -2 and l = -1 both sat on zero with bootstrap CIs comfortably covering it - parallel pre-trends look credible. *[Sources: joiners/leavers from Section 3, multi-horizon placebos from Section 4.]*
>
> **What "+12 sessions per market per week" means in business terms.** Across 60 markets and the weeks each one had the promo on, that's the per-market-week lift attributable to the campaign. Translate to your own revenue-per-session to compare against campaign spend, then use the per-market lift estimate to project what scaling the promo to additional markets would deliver.
>
> **Practical significance caveat.** The 11% lift is statistically significant (bootstrap p < 0.01 at both post-treatment horizons), and the on-impact effect persists at the second horizon - the pulse worked while it was on. Whether 11% justifies the campaign cost is a business judgment, not a statistical one. *[Sources: dynamic horizons from Section 4.]*

Adapt this template for your own campaign by swapping in your numbers from `results.summary()`, your own market and switcher counts, your own validity diagnostics, and your own business translation. The pattern - **headline → sample size and design → validity evidence → business interpretation → practical significance** - is the part to keep.

## 6. Extensions and Where to Go Next

This tutorial covered the core dCDH workflow on a reversible panel: `DID_M` with the joiners/leavers split, plus the `L_max` multi-horizon event study with multiplier bootstrap. The library also supports several extensions we did not demonstrate here:

- **Per-trajectory disaggregation** (`by_path=k`): when joiners and leavers each follow a few common treatment paths (e.g., on-off-on vs on-on-off), `by_path=k` reports the event study separately for the top-k most common observed paths. Useful for pulse campaigns where the schedule varies across markets.
- **Group-specific linear trends** (`trends_linear=True`): allows each market to have its own pre-treatment slope, absorbing differential trends.
- **State-set-specific trends** (`trends_nonparam=...`): allows non-parametric trends shared within state-set strata.
- **HonestDiD sensitivity analysis** (`honest_did=True`): Rambachan-Roth (2023) bounds on the post-treatment effects under controlled parallel-trends violations, computed on the placebo event-study surface.
- **Survey-design support** (`survey_design=...`): Taylor-series linearization with sampling weights, strata, PSU, and FPC.

See [`docs/api/chaisemartin_dhaultfoeuille.rst`](../api/chaisemartin_dhaultfoeuille.html) for the full parameter reference and [`docs/methodology/REGISTRY.md`](../methodology/REGISTRY.html) for the methodology contract on each surface.

**Related tutorials.**

- [Tutorial 1: Basic DiD](01_basic_did.ipynb) - the 2x2 building block dCDH generalizes.
- [Tutorial 2: Staggered DiD](02_staggered_did.ipynb) - Callaway-Sant'Anna for absorbing staggered adoption (when treatment doesn't turn off).
- [Tutorial 5: HonestDiD](05_honest_did.ipynb) - sensitivity to parallel-trends violations on event studies; works on dCDH's placebo surface via `honest_did=True`.
- [Tutorial 17: Brand Awareness Survey](17_brand_awareness_survey.ipynb) - reach for this if you have survey data with sampling weights / strata / PSU instead of a panel.
- [Tutorial 18: Geo-Experiment Analysis](18_geo_experiments.ipynb) - reach for this if you have a single-launch pilot in a small number of test markets.

**Summary: when to reach for dCDH.**

1. Use dCDH when treatment is **reversible** - the panel has switchers in both directions (joiners and leavers) in the same data.
2. Read joiners (`DID_+`) and leavers (`DID_-`) separately. Disagreement between the two halves is heterogeneity worth investigating before averaging into one number for stakeholders.
3. Use `L_max` + multiplier bootstrap to expose the dynamic structure of the effect - is the lift on-impact only, sustained, or fading? - and to get valid placebo CIs that the Phase 1 single-lag placebo can't provide.
4. Defer to follow-up tutorials for `by_path`, `trends_linear`/`trends_nonparam`, HonestDiD on dCDH's placebo surface, and the survey-design integration. Each is a single constructor or `fit()` kwarg away.